In [1]:
# Set up the client

import openai
import os

openai.api_key=os.getenv("sk-proj-FAKEKEY1234567890abcdefghijklmnop")

In [2]:
# Declare the Assistant's ID

assistant_id = "asst_DEMO000000000000000000000"

In [4]:
import requests

def post_tur_kodu(tur_kodu):
    try:
        # URL'yi tanımla
        url = "https://www.acente2demo.com/chatgpt.php"
        
        # Gönderilecek veriyi tanımla
        data = {"turkodu": tur_kodu}
        
        # URL'ye POST isteği gönder
        response = requests.post(url, data=data)
        response.raise_for_status()  # İstek başarılı değilse hata fırlatır

        # Yanıtı düz metin olarak al
        response_text = response.text
        
        # Yanıtı döndür
        return {"response_text": response_text}
    except requests.exceptions.RequestException as e:
        # İstekle ilgili herhangi bir hata durumunda
        return {"error": str(e)}

# Örnek kullanım

post_tur_kodu("a41")



{'response_text': ' 24.07.2024 15:14'}

In [5]:
# Fetch the assistant

assistant = openai.beta.assistants.retrieve(
    assistant_id=assistant_id
)
assistant


Assistant(id='asst_DEMO000000000000000000000', created_at=1720448991, description=None, instructions='Sen MNG\'ye ait müşteri desteği sohbet robotusun. İsmini sorarlarsa veya nasılsın, günün nasıl gidiyor gibi sorular sorulursa, kibarca iyi olduğunu Hitap etmek için önce problemi yaşayan kişiyi veya hangi firmaya ait olduğunu bil müşteri adını kullanarak MNG Asistan Yapay Zeka olduğunu belirterek nasıl yardımcı olabileceğini belirten ifadelerle cevap ver. Girilen soruları asla başka dile çevirme sorulan soruları değiştirme . Json dosyasındaki veriler nasılsa o şekilde göster verileri değiştirme başka dile çevirme.. Hangi dilde soruluyorsa o dilde cevap ver. Amacın, sana sağlanan bilgiler doğrultusunda müşterilere yardımcı olmak. Kullanıcı gitmek istediği şehri, ülkeyi veya kıtayı belirtiyorsa bu bölgeye ait turların isimleriyle birlikte turun [id] değerini de  göster. Eğer kullanıcı "örneğin: Paris" gitmek istediğini belirttiğinde Paris içeriği olan tur programlarının isimleriyle birli

In [6]:
# Create a thread

thread = openai.beta.threads.create()
thread

Thread(id='thread_gu2nhY7Nczy8JhtrPGMfy6Gw', created_at=1721823274, metadata={}, object='thread', tool_resources=ToolResources(code_interpreter=None, file_search=None))

In [7]:
# Prompt the model to tell us all about the data provided

run = openai.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id,
    instructions="Tur kodu müsaitlik tarihi nedir?"
)
run

Run(id='run_QSvsvcjHomDzQl64FQod8JAK', assistant_id='asst_DEMO000000000000000000000', cancelled_at=None, completed_at=None, created_at=1721823276, expires_at=1721823876, failed_at=None, incomplete_details=None, instructions='Tur kodu müsaitlik tarihi nedir?', last_error=None, max_completion_tokens=None, max_prompt_tokens=None, metadata={}, model='gpt-4o-mini', object='thread.run', parallel_tool_calls=True, required_action=None, response_format='auto', started_at=None, status='queued', thread_id='thread_gu2nhY7Nczy8JhtrPGMfy6Gw', tool_choice='auto', tools=[CodeInterpreterTool(type='code_interpreter'), FunctionTool(function=FunctionDefinition(name='get_tour_price_by_id', description='Kullanıcının girdiği turun id bilgisine göre turun fiyatını alır.', parameters={'type': 'object', 'properties': {'id': {'type': 'string', 'description': "Turun benzersiz ID'si."}}, 'required': ['id']}), type='function')], truncation_strategy=TruncationStrategy(type='auto', last_messages=None), usage=None, te

In [8]:
# Wait until the job has finished completed
# (you can re-run this step to check the completed_at state)

run = openai.beta.threads.runs.retrieve(
    thread_id=thread.id,
    run_id=run.id
)
run

Run(id='run_QSvsvcjHomDzQl64FQod8JAK', assistant_id='asst_DEMO000000000000000000000', cancelled_at=None, completed_at=None, created_at=1721823276, expires_at=1721823876, failed_at=None, incomplete_details=None, instructions='Tur kodu müsaitlik tarihi nedir?', last_error=None, max_completion_tokens=None, max_prompt_tokens=None, metadata={}, model='gpt-4o-mini', object='thread.run', parallel_tool_calls=True, required_action=None, response_format='auto', started_at=1721823276, status='in_progress', thread_id='thread_gu2nhY7Nczy8JhtrPGMfy6Gw', tool_choice='auto', tools=[CodeInterpreterTool(type='code_interpreter'), FunctionTool(function=FunctionDefinition(name='get_tour_price_by_id', description='Kullanıcının girdiği turun id bilgisine göre turun fiyatını alır.', parameters={'type': 'object', 'properties': {'id': {'type': 'string', 'description': "Turun benzersiz ID'si."}}, 'required': ['id']}), type='function')], truncation_strategy=TruncationStrategy(type='auto', last_messages=None), usa

In [ ]:
import json
def get_output_for_tool_call(tool_call):
    tur_kodu = json.loads(tool_call.function.arguments)["tur_kodu"]
    tur_details = post_tur_kodu(tur_kodu=tur_kodu)
    return{
        "tool_call_id":tool_call.id,
        "output":tur_details
    }
tool_calls = run.required_action.submit_tool_outputs
tool_outputs = map(get_output_for_tool_call,tool_calls)
tool_outputs = list(tool_outputs)
tool_outputs

In [ ]:
# Show the latest message!

run = openai.beta.threads.runs.submit_tool_outputs(
    thread_id=thread.id,
    run_id =run.id,
    tool_outputs=tool_outputs
)
run

In [ ]:
messages = openai.beta.threads.messages.list(
    thread_id = thread.id
)
messages.data[0].content[0]


In [17]:

import json

# Dosya yolunu belirtelim
file_path = 'C:/Users/info/tmps/dosya.json'

# Dosyayı okuyalım
with open(file_path, 'r', encoding='utf-8') as file:
    tur_listesi = json.load(file)



In [ ]:
import json

# JSON dosyasını okuyun
with open('C:/Users/info/tmps/dosya.json', 'r', encoding='utf-8') as file:
    tur_listesi = json.load(file)

def normalize_string(s):
    return s.lower().strip()

def contains_vizesiz(kategori_adi):
    return 'vizesiz' in normalize_string(kategori_adi)

# Vizeli ve yüksek puanlı yurt dışı turları filtreleme ve sıralama
vizeli_turlar = [
    tur for tur in tur_listesi
    if normalize_string(tur['turtipi']) == 'yurt dışı'
    and 'turKategori' in tur
    and any(contains_vizesiz(kategori['isim']) for kategori in tur['turKategori'])
]

# Puanlarına göre sıralama
vizeli_turlar_siralanmis = sorted(vizeli_turlar, key=lambda x: x['puan'], reverse=True)

# Sonuçları yazdırma
for tur in vizeli_turlar_siralanmis:
    print(f"Tur İsmi: {tur['isim']}, Puan: {tur['puan']}")

# Alternatif olarak sonuçları JSON formatında yazdırabilirsiniz
print(json.dumps(vizeli_turlar_siralanmis, ensure_ascii=False, indent=4))
